# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`

This notebook demonstrates how to use the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library to explore, process, and visualize a dataset defined by a Croissant schema. We focus on robust, reproducible data access using semantic IDs (`@id`) as per the Croissant and FAIR guidelines.

## Dataset Source
The dataset is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset's metadata and inspect basic information using `mlcroissant`. We'll use the dataset URL as the entry point.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Instantiate the Dataset
ds = mlc.Dataset(croissant_url)

# Display dataset-level metadata
meta = ds.metadata
print(f"{meta.name}: {meta.description}\n")

## 2. Data Overview
Review available record sets and fields. We'll list all record sets and their `@id`s, along with fields and columns available in the main record set.

Croissant datasets organize actual data into **record sets**, each having a stable semantic `@id`. We'll print all available record set IDs and field IDs, as well as get the first few records for further inspection.

In [ ]:
# List all available record sets and their field IDs
record_sets = ds.metadata.record_sets

print("Available record sets and fields:")
for rs in record_sets:
    print(f"- Record Set @id: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"    - Field @id: {field['@id']} (name: {field.get('name', field['@id'])})")

# For demonstration, pick the first record set for a preview
if len(record_sets) == 0:
    raise ValueError('No record sets found in this dataset.')
first_record_set_id = record_sets[0]['@id']
print(f"\nFirst 3 records from Record Set {first_record_set_id}:")
for idx, rec in enumerate(ds.records(record_set=first_record_set_id)):
    print(rec)
    if idx >= 2:
        break

## 3. Data Extraction
Let's load the principal record set into a DataFrame for downstream processing. All referencing is done by `@id`.

We'll load **all** available record sets into separate DataFrames for inspection.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in ds.metadata.record_sets]
dataframes = {}

for rsid in record_set_ids:
    records = list(ds.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)

selected_record_set_id = record_set_ids[0]  # Selecting the main record set
print(f"Loaded columns in record set {selected_record_set_id}:")
print(dataframes[selected_record_set_id].columns.tolist())

dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's explore, filter, and process numeric data. We'll identify a numeric field using the fields listed. For demonstration, we'll look for a numeric field, such as age or interval (replace as appropriate depending on field names in your dataset).

**Note:** Replace the `numeric_field_id` and `group_field_id` below with actual `@id` values appropriate for the dataset (see section 2 for available field IDs and column names for guidance).

In [ ]:
# Identify a numeric field @id for analysis ('cr:interval_between_cancers' is a plausible example):
# Update these with real @ids as shown in section 2 output.
numeric_field_id = None  # e.g., 'cr:Age_at_second_CRC' or similar
group_field_id = None    # e.g., 'cr:Sex' or 'cr:Anatomical_location_second_CRC'
main_rs_id = selected_record_set_id

# Auto-select a numeric column
df = dataframes[main_rs_id]
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    raise ValueError('No numeric field detected. Edit "numeric_field_id" as appropriate.')

# Try to auto-select a group field (@id) that's low cardinality categorical
for col in df.columns:
    if df[col].dtype == object and df[col].nunique() < 15:
        group_field_id = col
        break
if group_field_id is None:
    # Default to first column if none found
    group_field_id = df.columns[0]

print(f"Using numeric field: {numeric_field_id}")
print(f"Grouping field: {group_field_id}")

# EDA: filtering, normalizing, grouping
threshold = df[numeric_field_id].mean()  # Example: use mean as threshold
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count'])
    print(f"\nGrouped statistics by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization

Visualize distributions or relationships of fields (using their `@id`/column names). We'll plot the numeric field's distribution and a grouped barplot.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id], kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

if group_field_id in df.columns:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion

- Demonstrated exploration of FAIR²-compliant clinical dataset using stable `@id` references.
- Inspected available record sets and fields, loaded all data, and performed simple EDA and visualization on a numeric field.
- All references (record sets, fields) are by `@id`, ensuring schema-robust coding.

You may further customize field and group selection, apply domain-specific cleaning, and connect this notebook to ML pipelines or standard dashboards.